In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib
import re
import os
from datetime import datetime

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
 
# Models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
 
# Model selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
)
 
# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
# Feature importance
from sklearn.inspection import permutation_importance


In [2]:
CONFIG = {
    "input": "Outputs/lr_epc_towns.parquet",
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 5,
    "n_jobs": -1,
 
    # Columns to drop (merge keys, address fields, intermediate columns)
    "drop_columns": [
        "transaction_id",       # Identifier

        "postcode_x",           # Postcodes — five versions
        "postcode_clean", 
        "postcode_merge",
        "PCDS", 
        "postcode_y",

        "address_key",          # Address components 
        "address1",
        "paon", 
        "saon", 
        "street", 
        "locality", 
        "town", 
        "district", 
        "county",

        "record_status",        # Admin fields -> Land Registry internal codes, not property attributes
        "ppd_category",

                                # Pipeline fields — served their purpose during data joining
        "uprn",                 # used to get exact coordinates
        "lodgement_date",       # EPC lodgement date, not the sale date
        
                                # Coordinate fallbacks — keeping exact_lat/exact_lon, dropping centroid
        "LAT",                  # postcode centroid latitude (less precise)
        "LONG",                 # postcode centroid longitude (less precise)
        "has_exact_coords",     # flag column, not a feature
        
        "dist_bus_km",          # Bus Features -> I dont want them right now
        "bus_within_1km",

        "nearest_town",         # Town name — keeping only dist_town_km, since the name is one of 112 categories
        
        "mains_gas_flag"        # Dropping as 1/4 is missing this flag
        
    ],
}

In [3]:
print("Loading dataset...")
df = pd.read_parquet(CONFIG["input"])
print(f"  Raw: {len(df):,} rows, {len(df.columns)} columns")

# Drop rows with no EPC (no house details)
before = len(df)
df = df[df["total_floor_area"].notna()]
print(f"  Has EPC: {before:,} → {len(df):,} (dropped {before - len(df):,})")

# Drop rows with no coordinates (those have no distances)
before = len(df)
df = df[df["dist_town_km"].notna()]
print(f"  Has coords: {before:,} → {len(df):,} (dropped {before - len(df):,})")

# Drop useless columns
df = df.drop(columns=CONFIG["drop_columns"], errors="ignore")

# Extract date features
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df = df.drop(columns=["date"])

# Clean up construction_age_band to remove "England and Wales: " prefix, if present
df["construction_age_band"] = df["construction_age_band"].str.replace("England and Wales: ", "", regex=False)

# Remove commercial properties
before = len(df)
df = df[df["property_type_x"] != "O"]
print(f"  Removed type O: {before:,} → {len(df):,}")

# Remove low transactions - below 1000 - huge outliers. 
before = len(df)
df = df[df["price"] >= 1000]
df = df[df["price"] <= 5_000_000]
print(f"  Removed price < £1000: {before:,} → {len(df):,}")
print(f"  Removed price > £5,000,000: {before:,} → {len(df):,}")


before = len(df)
df = df[df["exact_lat"].notna()]
print(f"  Removed no coords: {before:,} → {len(df):,}")


before = len(df)
df = df[(df["total_floor_area"] >= 10) & (df["total_floor_area"] <= 1000)]
print(f"  Removed floor area < 10 or > 1000m²: {before:,} → {len(df):,}")

print(f"  After filtering: {len(df):,} rows, {len(df.columns)} columns")



Loading dataset...
  Raw: 5,890,089 rows, 53 columns
  Has EPC: 5,890,089 → 4,379,246 (dropped 1,510,843)
  Has coords: 4,379,246 → 4,379,226 (dropped 20)
  Removed type O: 4,379,226 → 4,307,758
  Removed price < £1000: 4,307,758 → 4,305,876
  Removed price > £5,000,000: 4,307,758 → 4,305,876
  Removed no coords: 4,305,876 → 4,298,607
  Removed floor area < 10 or > 1000m²: 4,298,607 → 4,298,291
  After filtering: 4,298,291 rows, 28 columns


In [4]:
# Check for null values
nans = df.isnull().sum()
print(nans[nans > 0])
print(f"\nTotal rows: {len(df):,}")

print(pd.crosstab(df["property_type_x"], df["property_type_y"]))

print(f"Rentals: {(df['transaction_type'] == 'rental').sum():,}")

print(df["total_floor_area"].describe())
print(f"Above 500m²: {(df['total_floor_area'] > 500).sum():,}")
print(f"Above 1000m²: {(df['total_floor_area'] > 1000).sum():,}")
print(f"Zero floor area: {(df['total_floor_area'] == 0).sum():,}")
print(f"Below 10m²: {(df['total_floor_area'] < 10).sum():,}")


number_habitable_rooms    666921
tenure                     65957
property_type_y               25
transaction_type             460
construction_age_band          2
built_form                 18844
main_fuel                  10050
dtype: int64

Total rows: 4,298,291
property_type_y  Bungalow    Flat    House  Maisonette  Park home
property_type_x                                                  
D                  223523    1577   868391         590         33
F                    1691  266980    16900       45862          1
S                  138827    5563  1283620        1988          1
T                   22325   12499  1403629        4265          1
Rentals: 0
count    4.298291e+06
mean     9.614869e+01
std      4.126970e+01
min      1.000000e+01
25%      7.100000e+01
50%      8.600000e+01
75%      1.100000e+02
max      1.000000e+03
Name: total_floor_area, dtype: float64
Above 500m²: 1,519
Above 1000m²: 0
Zero floor area: 0
Below 10m²: 0


In [5]:
# we dont want too much data initally 
df = df.sample(n=1_000_000, random_state=42) 

In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", None)
print(df.head().to_string())

          price property_type_x new_build duration  total_floor_area  current_energy_efficiency current_energy_rating  number_habitable_rooms            tenure property_type_y transaction_type construction_age_band     built_form                  main_fuel  exact_lat  exact_lon  dist_primary_km  dist_secondary_km  dist_rail_km  rail_within_1km  rail_within_5km  dist_metro_km  metro_within_1km  dist_airport_km  dist_coast_km  dist_town_km  sale_year  sale_month
2340769  108000               S         N        F              87.0                       65.0                     D                     5.0  rented (private)           House           Rental             1967-1975  Semi-Detached  mains gas (not community)  53.542092  -1.044541         1.082776           1.028398      3.109847              0.0              1.0      22.632978               0.0        46.190523       3.441841      5.734698       2020           8
771013   235000               F         N        L              52.0  

In [7]:
# =============================================================================
# SECTION 2.1: FEATURE CLEAN UP
# =============================================================================

def clean_age_band(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    
    # Already a known range — keep as is
    known_ranges = [
        "before 1900", "1900-1929", "1930-1949", "1950-1966",
        "1967-1975", "1976-1982", "1983-1990", "1991-1995",
        "1996-2002", "2003-2006", "2007-2011", "2012 onwards",
    ]
    if val in known_ranges:
        return val
    
    # Try to parse as a year
    try:
        year = int(val)
        if year < 1900: return "before 1900"
        elif year < 1930: return "1900-1929"
        elif year < 1950: return "1930-1949"
        elif year < 1967: return "1950-1966"
        elif year < 1976: return "1967-1975"
        elif year < 1983: return "1976-1982"
        elif year < 1991: return "1983-1990"
        elif year < 1996: return "1991-1995"
        elif year < 2003: return "1996-2002"
        elif year < 2007: return "2003-2006"
        elif year < 2012: return "2007-2011"
        else: return "2012 onwards"
    except:
        pass
    
    # Catch other range formats
    if "2007 onwards" in val or "2012-2021" in val:
        return "2012 onwards"
    
    # Everything else (letters, junk) — Unknown
    return "Unknown"

def clean_fuel(val):
    if pd.isna(val):
        return val
    val = val.lower()
    if "gas" in val and "lpg" not in val:
        return "mains_gas"
    elif "electric" in val:
        return "electricity"
    elif "oil" in val or "biodiesel" in val:
        return "oil"
    elif "lpg" in val or "lng" in val:
        return "lpg"
    elif "wood" in val or "biomass" in val or "biogas" in val:
        return "biomass"
    else:
        return "other"
    
def clean_transaction(val):
    if pd.isna(val):
        return val
    val = val.lower().strip()
    if "marketed sale" in val and "non" not in val:
        return "marketed_sale"
    elif "non" in val and ("marketed" in val or "sale" in val):
        return "non_marketed_sale"
    elif "new dwelling" in val:
        return "new_dwelling"
    elif "rental" in val or "rent" in val:
        return "rental"
    else:
        return "other"

def clean_rooms(df):
    """Remove clearly wrong room counts, fill missing based on floor area."""
    # Only remove extreme data errors (97, 100 rooms etc.)
    cap = df["number_habitable_rooms"].quantile(0.99)
    print(f"  Rooms: 99th percentile = {cap}, removing above")
    df.loc[df["number_habitable_rooms"] > cap, "number_habitable_rooms"] = np.nan
    
    # Fill missing based on median rooms for similar sized properties
    has_both = df["number_habitable_rooms"].notna() & df["total_floor_area"].notna()
    bins = [0, 30, 50, 70, 90, 120, 160, 200, 500, 10000]
    df["area_bin"] = pd.cut(df["total_floor_area"], bins=bins)
    median_rooms = df.loc[has_both].groupby("area_bin")["number_habitable_rooms"].median()
    
    missing = df["number_habitable_rooms"].isna()
    df.loc[missing, "number_habitable_rooms"] = df.loc[missing, "area_bin"].map(median_rooms)
    df = df.drop(columns=["area_bin"])
    
    filled = missing.sum() - df["number_habitable_rooms"].isna().sum()
    print(f"  Rooms: filled {filled:,} missing based on floor area")
    return df

df["construction_age_band"] = df["construction_age_band"].apply(clean_age_band)
df["main_fuel"] = df["main_fuel"].apply(clean_fuel)
df["transaction_type"] = df["transaction_type"].apply(clean_transaction)
df["tenure"] = df["tenure"].str.lower().str.strip()
df = clean_rooms(df)


  Rooms: 99th percentile = 9.0, removing above
  Rooms: filled 160,841 missing based on floor area


In [8]:
# =============================================================================
# SECTION 2: DEFINE FEATURES AND TARGET
# =============================================================================
 
target = "price"
 
numeric_features = [
    # EPC features
    "total_floor_area",           # floor area in m²
    "current_energy_efficiency",  # numeric efficiency score
    "number_habitable_rooms",     # number of rooms
 
    # Location — exact coordinates
    "exact_lat",                  # property latitude
    "exact_lon",                  # property longitude
 
    # Spatial — school distances
    "dist_primary_km",            # distance to nearest primary school
    "dist_secondary_km",          # distance to nearest secondary school
 
    # Spatial — transport distances and density
    "dist_rail_km",               # distance to nearest rail station
    "rail_within_1km",            # number of rail stations within 1km
    "rail_within_5km",            # number of rail stations within 5km
    "dist_metro_km",              # distance to nearest metro/underground
    "metro_within_1km",           # number of metro stations within 1km
    "dist_airport_km",            # distance to nearest airport
 
    # Spatial — geography
    "dist_coast_km",              # distance to nearest coastline
    "dist_town_km",               # distance to nearest major town (ONS 75k+)
 
    # Date
    "sale_year",                  # year of sale
    "sale_month",                 # month of sale
]
 
categorical_features = [
    "property_type_x",            # D=Detached, S=Semi, T=Terraced, F=Flat, O=Other
    "duration",                   # F=Freehold, L=Leasehold
    "current_energy_rating",      # A-G energy rating
    "tenure",                     # owner-occupied, rented, etc.
    "property_type_y",            # House, Flat, Bungalow (from EPC)
    "built_form",                 # Detached, Semi-Detached, Mid-Terrace, etc.
    "construction_age_band",      # e.g. England and Wales: 1900-1929
    "main_fuel",                  # mains gas, electricity, etc.
    "transaction_type",           # marketed sale, rental, etc.
]
 
binary_features = [
    "new_build",                  # Y/N — is it a new build
]

# Verify all features exist in the dataframe
all_features = numeric_features + categorical_features + binary_features
missing = [f for f in all_features if f not in df.columns]
if missing:
    print(f"  WARNING: missing columns: {missing}")
    # Remove missing from their lists so pipeline doesn't break
    numeric_features = [f for f in numeric_features if f in df.columns]
    categorical_features = [f for f in categorical_features if f in df.columns]
    binary_features = [f for f in binary_features if f in df.columns]
    all_features = numeric_features + categorical_features + binary_features
 
X = df[all_features]
y = df[target]
 
print(f"\n  Features: {len(all_features)}")
print(f"    Numeric:     {len(numeric_features)}")
print(f"    Categorical: {len(categorical_features)}")
print(f"    Binary:      {len(binary_features)}")
print(f"  Target:  {target} (mean=£{y.mean():,.0f}, median=£{y.median():,.0f})")


  Features: 27
    Numeric:     17
    Categorical: 9
    Binary:      1
  Target:  price (mean=£324,650, median=£265,000)


In [9]:
# Price distribution
print(df["price"].describe())
print(f"\n£0-100:        {(df['price'] <= 100).sum():,}")
print(f"£100-1000:     {(df['price'].between(101, 1000)).sum():,}")
print(f"£1k-10k:       {(df['price'].between(1001, 10_000)).sum():,}")
print(f"£10k-500k:     {(df['price'].between(10_001, 500_000)).sum():,}")
print(f"£500k-1M:      {(df['price'].between(500_001, 1_000_000)).sum():,}")
print(f"£1M-5M:        {(df['price'].between(1_000_001, 5_000_000)).sum():,}")
print(f"£5M-10M:       {(df['price'].between(5_000_001, 10_000_000)).sum():,}")
print(f"£10M+:         {(df['price'] > 10_000_000).sum():,}")

# Property type breakdown
print(f"\nProperty types:")
print(df["property_type_x"].value_counts())

# Type O prices
print(f"\nType O prices:")
print(df[df["property_type_x"] == "O"]["price"].describe())

# Categorical cardinality
print(f"\nUnique values per categorical:")
for col in categorical_features:
    print(f"  {col}: {df[col].nunique()}")

print(df["tenure"].value_counts())

count    1.000000e+06
mean     3.246503e+05
std      2.605254e+05
min      1.000000e+03
25%      1.750000e+05
50%      2.650000e+05
75%      3.940000e+05
max      5.000000e+06
Name: price, dtype: float64

£0-100:        0
£100-1000:     1
£1k-10k:       8
£10k-500k:     862,563
£500k-1M:      117,924
£1M-5M:        19,504
£5M-10M:       0
£10M+:         0

Property types:
property_type_x
T    335875
S    332211
D    254346
F     77568
Name: count, dtype: int64

Type O prices:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: price, dtype: float64

Unique values per categorical:
  property_type_x: 4
  duration: 2
  current_energy_rating: 7
  tenure: 4
  property_type_y: 5
  built_form: 7
  construction_age_band: 13
  main_fuel: 6
  transaction_type: 5
tenure
owner-occupied      741572
unknown             131682
rented (private)    102292
rented (social)       9319
Name: count, dtype: int64


In [10]:
print(df["construction_age_band"].value_counts())
print(df["main_fuel"].value_counts())
print(df["transaction_type"].value_counts())

construction_age_band
2012 onwards    159522
1950-1966       141716
1900-1929       137554
1930-1949       127602
1967-1975       104875
before 1900      73482
1983-1990        56333
1976-1982        50424
1996-2002        42475
2003-2006        30744
1991-1995        30742
Unknown          22629
2007-2011        21902
Name: count, dtype: int64
main_fuel
mains_gas      910285
electricity     54303
oil             21500
lpg              5655
other            3581
biomass          2390
Name: count, dtype: int64
transaction_type
marketed_sale        687234
new_dwelling         141770
rental               100099
other                 52493
non_marketed_sale     18307
Name: count, dtype: int64


In [11]:
# Verify cleaning worked
print(f"\nAfter cleaning:")
print(f"  Rows: {len(df):,}")
print(f"  construction_age_band: {df['construction_age_band'].nunique()} categories")
print(f"  main_fuel: {df['main_fuel'].nunique()} categories")
print(f"  transaction_type: {df['transaction_type'].nunique()} categories")
print(f"  Price range: £{df['price'].min():,.0f} — £{df['price'].max():,.0f}")
print(f"  Property types: {df['property_type_x'].unique()}")


After cleaning:
  Rows: 1,000,000
  construction_age_band: 13 categories
  main_fuel: 6 categories
  transaction_type: 5 categories
  Price range: £1,000 — £5,000,000
  Property types: <ArrowStringArray>
['S', 'F', 'T', 'D']
Length: 4, dtype: str


In [12]:
# =============================================================================
# SECTION 3: TRAIN/TEST SPLIT
# =============================================================================
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
)
 
print(f"\n  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")


  X_train: (800000, 27)
  X_test:  (200000, 27)


In [13]:
# =============================================================================
# SECTION 4: PREPROCESSOR
# =============================================================================
 
# Numeric: fill NaN with median, then scale
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
 
# Categorical: fill NaN with "Unknown", then one-hot encode
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
 
# Binary: fill NaN with most frequent, then ordinal encode
binary_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder()),
])
 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("bin", binary_transformer, binary_features),
    ]
)

In [14]:
# =============================================================================
# SECTION 5: BUILD PIPELINES
# =============================================================================

pipe_ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge())
])

pipe_lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Lasso(max_iter=1000)) # this should be upped for final traning to 5k
])

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42, n_jobs=-1, tree_method="hist"))
])

pipe_mlp = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", MLPRegressor(random_state=42, max_iter=500, early_stopping=True))
])

In [15]:
# =============================================================================
# SECTION 6: PARAM GRIDS FOR GRIDSEARCH
# =============================================================================
 
# Start with small grids — expand once you know what works

param_grids = {
    "Ridge": {
        "pipeline": pipe_ridge,
        "params": {
            "regressor__alpha": [1.0, 10.0, 100.0],
        }
    },
    "Lasso": {
        "pipeline": pipe_lasso,
        "params": {
            "regressor__alpha": [ 0.001, 0.01, 0.1],
        }
    },
    "XGBoost": {
        "pipeline": pipe_xgb,
        "params": {
            "regressor__n_estimators": [200, 300, 500],
            "regressor__max_depth": [6, 8, 10],
            "regressor__learning_rate": [0.05, 0.1],
            "regressor__subsample": [0.8],
            "regressor__colsample_bytree": [0.8],
        }
    },
    "RandomForest": {
        "pipeline": pipe_rf,
        "params": {
            "regressor__n_estimators": [100, 200],
            "regressor__max_depth": [10, 20, None],
            "regressor__min_samples_leaf": [5, 10],
        }
    }

}

"""

param_grids = {
    "Ridge": {
        "pipeline": pipe_ridge,
        "params": {
            "regressor__alpha": [0.1, 1.0, 10.0, 100.0],
        }
    },
    "Lasso": {
        "pipeline": pipe_lasso,
        "params": {
            "regressor__alpha": [0.0001, 0.001, 0.01, 0.1],
        }
    },
    "RandomForest": {
        "pipeline": pipe_rf,
        "params": {
            "regressor__n_estimators": [100, 200],
            "regressor__max_depth": [10, 20, None],
            "regressor__min_samples_leaf": [5, 10],
        }
   # },
    "XGBoost": {
        "pipeline": pipe_xgb,
        "params": {
            "regressor__n_estimators": [200, 300, 500],
            "regressor__max_depth": [6, 8, 10],
            "regressor__learning_rate": [0.05, 0.1],
            "regressor__subsample": [0.8],
            "regressor__colsample_bytree": [0.8],
        }
    },
    "MLP": {
        "pipeline": pipe_mlp,
        "params": {
            "regressor__hidden_layer_sizes": [(100,), (100, 50), (200, 100)],
            "regressor__alpha": [0.0001, 0.001, 0.01],
            "regressor__learning_rate_init": [0.001, 0.01],
        }
    },
}

"""

'\n\nparam_grids = {\n    "Ridge": {\n        "pipeline": pipe_ridge,\n        "params": {\n            "regressor__alpha": [0.1, 1.0, 10.0, 100.0],\n        }\n    },\n    "Lasso": {\n        "pipeline": pipe_lasso,\n        "params": {\n            "regressor__alpha": [0.0001, 0.001, 0.01, 0.1],\n        }\n    },\n    "RandomForest": {\n        "pipeline": pipe_rf,\n        "params": {\n            "regressor__n_estimators": [100, 200],\n            "regressor__max_depth": [10, 20, None],\n            "regressor__min_samples_leaf": [5, 10],\n        }\n   # },\n    "XGBoost": {\n        "pipeline": pipe_xgb,\n        "params": {\n            "regressor__n_estimators": [200, 300, 500],\n            "regressor__max_depth": [6, 8, 10],\n            "regressor__learning_rate": [0.05, 0.1],\n            "regressor__subsample": [0.8],\n            "regressor__colsample_bytree": [0.8],\n        }\n    },\n    "MLP": {\n        "pipeline": pipe_mlp,\n        "params": {\n            "regres

In [16]:
# Quick test — no log transform, no GridSearch

pipe_quick = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

pipe_quick.fit(X_train, y_train)
print(f"Train R²: {pipe_quick.score(X_train, y_train):.4f}")
print(f"Test R²:  {pipe_quick.score(X_test, y_test):.4f}")


Train R²: 0.6304
Test R²:  0.6315


In [17]:
# =============================================================================
# SECTION 7: TRAIN AND EVALUATE
# =============================================================================


cv = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])
 
results = {}
 
for name, config in param_grids.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
 
    start = time.time()
 
    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=CONFIG["n_jobs"],
        verbose=1,
    )
 
    grid.fit(X_train, y_train)
    train_time = time.time() - start
 
    # Predict
    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)
 
    # Metrics
    results[name] = {
        "best_params": grid.best_params_,
        "train_time": train_time,
        "train_rmse": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "test_rmse":  np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "train_mae":  mean_absolute_error(y_train, y_pred_train),
        "test_mae":   mean_absolute_error(y_test, y_pred_test),
        "train_r2":   r2_score(y_train, y_pred_train),
        "test_r2":    r2_score(y_test, y_pred_test),
        "model":      grid.best_estimator_,
        "train_mape": np.mean(np.abs((y_train - y_pred_train) / y_train)) * 100,
        "test_mape":  np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100,    
    }
 
    r = results[name]
    print(f"\n  Best params: {r['best_params']}")
    print(f"  Time: {r['train_time']:.1f}s")
    print(f"  Train — RMSE: £{r['train_rmse']:,.0f}, MAE: £{r['train_mae']:,.0f}, R²: {r['train_r2']:.4f}, MAPE: {r['train_mape']:.1f}%")
    print(f"  Test  — RMSE: £{r['test_rmse']:,.0f}, MAE: £{r['test_mae']:,.0f}, R²: {r['test_r2']:.4f}, MAPE: {r['test_mape']:.1f}%")
 



# =============================================================================
# SECTION 7.1: LOG RESULTS
# =============================================================================

log_file = "Outputs/experiment_log.csv"

for name, r in results.items():
    log_entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "model": name,
            "n_rows": len(df),
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": len(all_features),
            "best_params": str(r["best_params"]),
            "train_rmse": round(r["train_rmse"], 0),
            "test_rmse": round(r["test_rmse"], 0),
            "train_mae": round(r["train_mae"], 0),
            "test_mae": round(r["test_mae"], 0),
            "train_r2": round(r["train_r2"], 4),
            "test_r2": round(r["test_r2"], 4),
            "train_mape": round(r["train_mape"], 1),
            "test_mape": round(r["test_mape"], 1),
            "train_time": round(r["train_time"], 1),
            "cv_folds": CONFIG["cv_folds"],
    }
    log_df = pd.DataFrame([log_entry])

    if os.path.exists(log_file):
        log_df.to_csv(log_file, mode="a", header=False, index=False)
    else:
        log_df.to_csv(log_file, mode="w", header=True, index=False)

print(f"\nResults logged to {log_file}")



Training Ridge...
Fitting 5 folds for each of 3 candidates, totalling 15 fits

  Best params: {'regressor__alpha': 100.0}
  Time: 98.7s
  Train — RMSE: £158,357, MAE: £90,135, R²: 0.6304, MAPE: 34.9%
  Test  — RMSE: £158,236, MAE: £90,242, R²: 0.6315, MAPE: 34.7%

Training Lasso...
Fitting 5 folds for each of 3 candidates, totalling 15 fits


c:\Users\Asia\Desktop\UL\msc-house-price-prediction\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.606367e+14, tolerance: 5.428e+12
  model = cd_fast.enet_coordinate_descent(



  Best params: {'regressor__alpha': 0.1}
  Time: 2720.1s
  Train — RMSE: £158,357, MAE: £90,140, R²: 0.6304, MAPE: 34.9%
  Test  — RMSE: £158,237, MAE: £90,248, R²: 0.6315, MAPE: 34.7%

Training XGBoost...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

  Best params: {'regressor__colsample_bytree': 0.8, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 10, 'regressor__n_estimators': 500, 'regressor__subsample': 0.8}
  Time: 2376.4s
  Train — RMSE: £47,066, MAE: £31,824, R²: 0.9674, MAPE: 12.7%
  Test  — RMSE: £82,537, MAE: £42,703, R²: 0.8997, MAPE: 15.1%

Training RandomForest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

  Best params: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 5, 'regressor__n_estimators': 200}
  Time: 31111.3s
  Train — RMSE: £59,351, MAE: £28,586, R²: 0.9481, MAPE: 10.2%
  Test  — RMSE: £90,369, MAE: £46,140, R²: 0.8798, MAPE: 16.2%

Results logged to Outputs/experiment_log.csv


In [18]:
# =============================================================================
# SECTION 8: COMPARISON TABLE
# =============================================================================
 
print(f"\n{'='*50}")
print("MODEL COMPARISON")
print(f"{'='*50}")
 
comparison = pd.DataFrame({
    name: {
        "Train RMSE": f"£{r['train_rmse']:,.0f}",
        "Test RMSE":  f"£{r['test_rmse']:,.0f}",
        "Train MAE":  f"£{r['train_mae']:,.0f}",
        "Test MAE":   f"£{r['test_mae']:,.0f}",
        "Train MAPE":  f"{r['train_mape']:,.0f}%",
        "Test MAPE":   f"{r['test_mape']:,.0f}%",
        "Train R²":   f"{r['train_r2']:.4f}",
        "Test R²":    f"{r['test_r2']:.4f}",
        "Time (s)":   f"{r['train_time']:.1f}",
    }
    for name, r in results.items()
}).T
 
print(comparison.to_string())


MODEL COMPARISON
             Train RMSE Test RMSE Train MAE Test MAE Train MAPE Test MAPE Train R² Test R² Time (s)
Ridge          £158,357  £158,236   £90,135  £90,242        35%       35%   0.6304  0.6315     98.7
Lasso          £158,357  £158,237   £90,140  £90,248        35%       35%   0.6304  0.6315   2720.1
XGBoost         £47,066   £82,537   £31,824  £42,703        13%       15%   0.9674  0.8997   2376.4
RandomForest    £59,351   £90,369   £28,586  £46,140        10%       16%   0.9481  0.8798  31111.3


In [19]:
# =============================================================================
# SECTION 9: FIND BEST MODEL
# =============================================================================
 
best_name = max(results, key=lambda k: results[k]["test_r2"])
best_model = results[best_name]["model"]
 
print(f"\nBest model: {best_name} (Test R² = {results[best_name]['test_r2']:.4f})")
 
# Save best model
joblib.dump(best_model, "Outputs/best_model.joblib")
print(f"Saved to Outputs/best_model.joblib")


Best model: XGBoost (Test R² = 0.8997)
Saved to Outputs/best_model.joblib


In [20]:
# =============================================================================
# SECTION 10: FEATURE IMPORTANCE
# =============================================================================
 
print(f"\nComputing permutation importance for {best_name}...")
 
result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="r2",
    n_repeats=10,
    random_state=CONFIG["random_state"],
    n_jobs=CONFIG["n_jobs"],
)
 
# Get feature names after preprocessing
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
 
perm_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": result.importances_mean,
}).sort_values("Importance", ascending=False)
 
print("\nTop 20 features:")
print(perm_df.head(20).to_string(index=False))
 
# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top = perm_df.head(20)
ax.barh(top["Feature"], top["Importance"])
ax.set_xlabel("Mean decrease in R²")
ax.set_title(f"Permutation Feature Importance: {best_name}")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("Outputs/feature_importance.png", dpi=150, bbox_inches="tight")
print("Saved feature_importance.png")


Computing permutation importance for XGBoost...


ValueError: All arrays must be of the same length